# Day 4 Part 2 — Alignment and Variants in Python
## Beginner Edition · Day 4 (Saturday, May 2)

**Course:** Introduction to Bioinformatics — HCT Cohort
**Instructor:** Norah AlGhamdi

---

### What you'll do
In Part 1, you used Clustal Omega and ClinVar in your browser. Now we do the same things in Python — small, gentle steps.

By the end you will:

- 🧬 Align multiple sequences in Python
- 🔍 Find conserved positions across all sequences
- 📂 Read a real VCF file
- 📋 Filter variants by quality

### How to use this notebook
1. **Save your own copy:** `File → Save a copy in Drive`
2. **Run each cell with `Shift + Enter`**
3. Most cells are written for you — just run them.
4. **✏️ Your Turn** = a small change for you to make.

**No big code anywhere.** Each cell does one tiny thing. 🐢


---
## Setup — Install Biopython


In [ ]:
!pip install biopython --quiet


In [ ]:
from Bio import SeqIO, AlignIO
from Bio.Seq import Seq
print("Ready!")


---
# Section A — Multiple Sequence Alignment in Python


---
## Step A1 — Create our 4 insulin sequences

Same 4 species you aligned on the web. Let's save them as a FASTA file.


In [ ]:
fasta_text = """>Human_INS
ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGAC
>Mouse_Ins1
ATGGCCCTGTTGGTGCTGTTCCTGCCCCTGCTGGCCCTGCTTGCCCTCTGGGAGCCCAAG
>Cow_INS
ATGGCCCTGTGGACACGCCTCCTGCCCCTGCTGGCCCTGCTGGCGCTCTGGCCCCCAGCC
>Chicken_INS
ATGGCTCTCTGGATCCGATCACTGCCTCTTCTGGCCCTGCTTGTCTTCTCAGGGCCAGGA"""

with open("insulin_4species.fasta", "w") as f:
    f.write(fasta_text)

print("Saved insulin_4species.fasta")


---
## Step A2 — Look at the file we just saved

Read the records back and check we have all 4 species.


In [ ]:
for record in SeqIO.parse("insulin_4species.fasta", "fasta"):
    print("Species:", record.id, "  Length:", len(record.seq))


---
### 🧩 Try It Yourself — A2 Challenge: GC Content

You just printed the species name and length. Now extend that loop to also print the **GC content** (% of bases that are G or C) for each species.

**Expected output format:**
```
Human_INS     length=62   GC=58.1%
Mouse_Ins1    length=63   GC=56.3%
...
```

<details>
<summary>💡 Hint</summary>

- GC% = (count of G + count of C) / total length × 100  
- `str(record.seq).count('G')` counts G bases  
- Round with `round(value, 1)`
</details>


In [ ]:
# ✏️ YOUR CODE HERE
# Loop over insulin_4species.fasta and print name, length, AND GC%



---
## Step A3 — Submit to Clustal Omega from Python

Biopython has a tool that talks to the EMBL-EBI Clustal Omega server (the same one you used in Part 1).


In [ ]:
import requests

# Read our FASTA file as text
with open("insulin_4species.fasta") as f:
    sequences = f.read()

# Submit to EMBL-EBI Clustal Omega
submit = requests.post(
    "https://www.ebi.ac.uk/Tools/services/rest/clustalo/run",
    data={
        "email": "your_email@example.com",
        "sequence": sequences,
        "stype": "dna"
    }
)
job_id = submit.text
print("Job ID:", job_id)
print("Submitted! Now we wait for the alignment...")


---
## Step A4 — Wait for the alignment to finish

Check the status of the job. Run this cell repeatedly until you see `FINISHED`.


In [ ]:
import time

# Check job status
status = requests.get(f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/status/{job_id}").text
print("Status:", status)

# If RUNNING, wait a few seconds and run this cell again
# If FINISHED, move to the next step


*If status is `RUNNING`, wait 10 seconds and run the cell above again. Repeat until you see `FINISHED`.*


---
## Step A5 — Get the alignment result

Once the job has finished, download the alignment.


In [ ]:
result = requests.get(f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/result/{job_id}/aln-clustal_num").text

with open("insulin_alignment.aln", "w") as f:
    f.write(result)

print("Alignment saved! Here it is:")
print(result)


**Look at the bottom row** of the alignment — it shows where positions are conserved (`*` for identical) across all 4 species.

*This is the same output you saw in Part 1, but now generated from Python.*


---
## Step A6 — Count the conserved positions

How many positions are identical across ALL 4 species?


In [ ]:
# Read the alignment with Biopython
alignment = AlignIO.read("insulin_alignment.aln", "clustal")

print("Number of sequences:", len(alignment))
print("Alignment length:   ", alignment.get_alignment_length())

# Count fully conserved columns
identical_count = 0
for i in range(alignment.get_alignment_length()):
    column = alignment[:, i]
    if len(set(column)) == 1:  # all the same
        identical_count += 1

percent = identical_count / alignment.get_alignment_length() * 100
print(f"Identical columns: {identical_count} / {alignment.get_alignment_length()} ({percent:.1f}%)")


**That percentage is a real biological measurement** — it tells you how conserved insulin is across these species. Highly conserved genes are usually the ones that are most essential for life.


---
### 🧩 Try It Yourself — A6 Challenge: Longest Conserved Stretch

You counted how many positions are conserved in total. Now find the **longest consecutive run** of conserved positions.

Print:
- The length of the longest stretch
- Its start and end position (0-based index)

<details>
<summary>💡 Hint</summary>

- Use a `current_streak` counter that increases when a column is conserved, resets to 0 when it's not  
- Track `best_streak` and the `best_start` position  
- A column is conserved when `len(set(alignment[:, i])) == 1`
</details>


In [ ]:
# ✏️ YOUR CODE HERE
# Find the longest consecutive stretch of conserved alignment columns

alignment = AlignIO.read("insulin_alignment.aln", "clustal")



---
# Section B — Working with VCF Files

Now we shift to variants — the differences between individuals.


---
## Step B1 — Create a tiny VCF file

Normally VCF files come from variant calling. We'll write a small one ourselves so we have something to read.


In [ ]:
vcf_text = """##fileformat=VCFv4.2
##INFO=<ID=GENE,Number=1,Type=String,Description="Gene affected">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr17\t43071077\trs80357713\tG\tA\t228.3\tPASS\tGENE=BRCA1
chr17\t43091434\trs80357906\tT\tC\t194.6\tPASS\tGENE=BRCA1
chr17\t43095899\trs80357567\tA\tG\t312.1\tPASS\tGENE=BRCA1
chr13\t32340301\trs28897672\tC\tT\t45.2\tPASS\tGENE=BRCA2
chr13\t32363533\trs80358981\tG\tA\t250.0\tPASS\tGENE=BRCA2
chr11\t5226774\trs334\tT\tA\t189.4\tPASS\tGENE=HBB"""

with open("tiny.vcf", "w") as f:
    f.write(vcf_text)

print("Saved tiny.vcf with 6 variants")


---
## Step B2 — Read the VCF (header lines)

VCF is just text. Let's open it and look at the header lines (those starting with `##`).


In [ ]:
with open("tiny.vcf") as f:
    for line in f:
        if line.startswith("##"):
            print(line.strip())


Header lines describe the file's format and what each field means.


---
## Step B3 — Find the column header

The single line starting with `#CHROM` tells you what each column is.


In [ ]:
with open("tiny.vcf") as f:
    for line in f:
        if line.startswith("#CHROM"):
            print("Column header:", line.strip())
            columns = line.strip().split("\t")
            print("\nColumns are:")
            for i, col in enumerate(columns):
                print(f"  [{i}] {col}")


---
## Step B4 — Print each variant

Now skip the header lines and print the actual variants.


In [ ]:
print(f"{\"CHROM\":<8} {\"POS\":<10} {\"ID\":<14} {\"REF\":<5} {\"ALT\":<5} {\"QUAL\":<8}")
print("-" * 60)

with open("tiny.vcf") as f:
    for line in f:
        if line.startswith("#"):
            continue   # skip header lines
        fields = line.strip().split("\t")
        chrom, pos, vid, ref, alt, qual = fields[0:6]
        print(f"{chrom:<8} {pos:<10} {vid:<14} {ref:<5} {alt:<5} {qual:<8}")


Each row is one variant. You're now reading a VCF file like a real bioinformatician.


---
### 🧩 Try It Yourself — B4 Challenge: Count Variants per Chromosome

You just printed every variant row. Now go further: count how many variants are on **each chromosome** and print a small summary.

**Expected output:**
```
chr17    3 variants
chr13    2 variants
chr11    1 variants
```

<details>
<summary>💡 Hint</summary>

- Use a dictionary: `counts = {}`  
- The chromosome is `fields[0]` after splitting on `'\t'`  
- `counts[chrom] = counts.get(chrom, 0) + 1` increments safely
</details>


In [ ]:
# ✏️ YOUR CODE HERE
# Count how many variants appear on each chromosome



---
## Step B5 — Filter by quality

Real variant analysis often filters out low-quality variants. Let's keep only variants with QUAL > 100.


In [ ]:
print("HIGH-QUALITY variants (QUAL > 100):")
print("-" * 60)

with open("tiny.vcf") as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        chrom, pos, vid, ref, alt, qual = fields[0:6]
        if float(qual) > 100:
            print(f"{chrom:<8} {pos:<10} {vid:<14} {ref}→{alt}   QUAL={qual}")


Notice that the variant in BRCA2 with QUAL=45.2 was filtered out — too low quality to trust.


---
### 🧩 Try It Yourself — B5 Challenge: Filter by Quality AND Gene

Step B5 filtered variants with QUAL > 100. Now write a filter that keeps only variants where **QUAL > 200 AND the gene is BRCA1**.

Print the position, REF→ALT change, and quality score for each match.

<details>
<summary>💡 Hint</summary>

- Combine two conditions with `and` inside your `if` statement  
- Check `float(qual) > 200` for quality  
- Check `'GENE=BRCA1' in fields[7]` for the gene  
</details>


In [ ]:
# ✏️ YOUR CODE HERE
# Print variants where QUAL > 200 AND gene is BRCA1



---
## Step B6 — Find variants in a specific gene

Look at only BRCA1 variants by checking the INFO field.


In [ ]:
print("BRCA1 variants only:")
print("-" * 60)

with open("tiny.vcf") as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        info = fields[7]
        if "GENE=BRCA1" in info:
            chrom, pos, vid, ref, alt, qual = fields[0:6]
            print(f"  Position: {chrom}:{pos}   {ref}→{alt}   ID: {vid}")


---
### 🧩 Try It Yourself — B6 Challenge: Variant Summary Table

Step B6 filtered for BRCA1. Now make a summary table showing **every gene** and how many variants it has.

**Expected output:**
```
Gene      Variants
--------  --------
BRCA1     3
BRCA2     2
HBB       1
```

<details>
<summary>💡 Hint</summary>

- Use a dictionary to count: `gene_counts = {}`  
- Parse the gene from the INFO field: `info.split('GENE=')[1]`  
- After the loop, iterate over the dictionary to print the table
</details>


In [ ]:
# ✏️ YOUR CODE HERE
# Build a summary table: gene name → number of variants



---
## Step B7 — Look up a variant in ClinVar by ID

Each variant has an `rs` ID — that's the dbSNP/ClinVar ID. We can look it up directly.


In [ ]:
rs_id = "rs334"   # this is a famous sickle cell variant

url = f"https://www.ncbi.nlm.nih.gov/clinvar/?term={rs_id}"
print("Open this URL in your browser:")
print(url)

print("\n(In Part 1, you searched ClinVar by gene. Now you can look up by variant ID directly.)"
)


---
### 🧩 Try It Yourself — B7 Challenge: Pairwise Sequence Identity

You've worked with alignment and variants separately. Now combine both ideas: calculate the **percent identity** between every pair of species in the insulin alignment.

Percent identity = positions where both sequences have the **same base** / alignment length × 100

**Example output:**
```
Human_INS  vs  Mouse_Ins1  →  85.5%
Human_INS  vs  Cow_INS     →  91.2%
...
```

<details>
<summary>💡 Hint</summary>

- Access sequences with `alignment[i].seq` and names with `alignment[i].id`  
- Use a nested loop: `for i in range(len(alignment))` → `for j in range(i+1, len(alignment))` to get all pairs without repeating  
- Count matches: `sum(a == b for a, b in zip(seq1, seq2))`
</details>


In [ ]:
# ✏️ YOUR CODE HERE
# Calculate pairwise percent identity for all species pairs

alignment = AlignIO.read("insulin_alignment.aln", "clustal")



---
## 📚 Homework — Apply what you learned


### Homework 1 — Count BRCA2 variants

From `tiny.vcf`, count how many variants are in BRCA2.


In [ ]:
count = 0
with open("tiny.vcf") as f:
    for line in f:
        if line.startswith("#"):
            continue
        if "GENE=BRCA2" in line:
            count += 1

print("Number of BRCA2 variants:", count)


### Homework 2 — How many positions are conserved?

From your insulin alignment, what fraction of positions had different bases across the 4 species?

(Hint: total length minus the conserved count, then divide.)


In [ ]:
alignment = AlignIO.read("insulin_alignment.aln", "clustal")
total = alignment.get_alignment_length()

# Count conserved (same in all 4)
conserved = 0
for i in range(total):
    column = alignment[:, i]
    if len(set(column)) == 1:
        conserved += 1

different = total - conserved
print(f"Total positions:     {total}")
print(f"Conserved positions: {conserved}")
print(f"Different positions: {different}")
print(f"Diversity:           {different/total*100:.1f}%")


### Homework 3 — Bonus 🎁

**Pick a gene that interests you. Add a 5th species to the alignment.**

1. Search NCBI for your gene in a different species (e.g., zebrafish, fruit fly, dog)
2. Download a small piece of the sequence
3. Add it to the FASTA from Step A1
4. Re-run the alignment
5. Bring the result to next session — which species is most similar to which?


---
## 🎉 You did it!

Today you learned to do — in Python — what bioinformaticians do every day:

| Skill | Real-world use |
|---|---|
| Multiple sequence alignment | Comparing genes across species |
| Find conserved positions | Identifying functionally important regions |
| Read a VCF file | Working with variant data |
| Filter variants by quality | Cleaning up noisy results |
| Look up variants in ClinVar | Connecting variants to disease |

### What's next — Weekend 3

Next weekend (May 8–9), we'll cover **RNA-seq** and **functional annotation** — connecting genes to what they actually DO in the cell.

**You finished Weekend 2.** You now have a real bioinformatics foundation. 🌟
